In [1]:
from openai import OpenAI

client = OpenAI()

In [2]:
import alpaca_trade_api as tradeapi

key = "PKMA3YUJIMEQVLWD5XTFZIXS3T"
secret_key = 'B8fe7qYHkR51PhrCFEn2UhAXYumfERAD1Vpb8jxx3v9V'
BASE_URL='https://paper-api.alpaca.markets'

api = tradeapi.REST(key, secret_key, BASE_URL, api_version='v2')

In [5]:
def fetch_portfolio():
  positions = api.list_positions()
  portfolio = []
  for pos in positions: 
    portfolio.append({
      'symbol': pos.symbol,
      'qty': pos.qty,
      'entry_price': pos.avg_entry_price,
      'current_price': pos.current_price,
      'unrealized_pl': pos.unrealized_pl,
      'side': 'buy'
    })
  return portfolio

def fetch_open_orders():
  orders = api.list_orders(status='open')
  open_orders = []
  for order in orders: 
    open_orders.append({
      'symbol': order.symbol,
      'qty': order.qty,
      'limit_price': order.limit_price,
      'side': 'buy'
    })
  return open_orders

def analyze_message(message):
  portfolio_data = fetch_portfolio()
  open_orders = fetch_open_orders()

  pre_prompt = f"""
  You are an AI Portfolio Manage responsible for analyzing my portfolio.
  Your tasks are the following:
  1.) Evaluate the risk exposures of my current holdings
  2.) Analyze my open limit orders and their potential impact
  3.) Provide insights into portfolio health, diversificaiton, trade adj. etc.
  4.) Speculate on the market outlook based on current market conditions
  5.) Identify Potential market risks and suggest risk management strategies

  Here is my Portfolio: {portfolio_data}

  Here are my Open Orders: {open_orders}

  Question: {message}
  """
  response = client.chat.completions.create(
    model = 'gpt-4o-mini',
    messages = [{"role": "system", "content": "You are a professional portfolio analyst." },
                {"role": "user", "content": pre_prompt}],
  )
  return response['choices'][0]['message']['content']

analysis = analyze_message("How is my portfolio?")
print(analysis)


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}